**Setup Data**

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce_orders
(
order_id INT,
customer_name STRING,
city STRING,
product STRING,
quantity INT,
price INT,
order_status STRING
)
USING DELTA;

**Inserting Values**

In [0]:
%sql
INSERT INTO ecommerce_orders VALUES
(101,'Amit Sharma','Hyderabad','Laptop',1,75000,'Placed'),
(102,'Priya Reddy','Bangalore','Mobile',2,30000,'Placed'),
(103,'Rohit Mehta','Mumbai','Headphones',3,2000,'Shipped'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Placed'),
(105,'Karan Patel','Ahmedabad','Tablet',2,25000,'Cancelled'),
(106,'Ananya Das','Kolkata','Mobile',1,28000,'Placed'),
(107,'Vikram Singh','Delhi','Camera',1,55000,'Placed'),
(108,'Meera Nair','Bangalore','Laptop',1,80000,'Placed');

num_affected_rows,num_inserted_rows
8,8


**PART 1 — INSERT Exercises**

In [0]:
insert into ecommerce_orders values
(109,'Rahul Verma','Mumbai','Tablet',1,27000,'Placed');

num_affected_rows,num_inserted_rows
1,1


In [0]:
insert into ecommerce_orders values
(110,'Arjun Nair','Kochi','Mobile',1,30000,'Placed'),
(111,'Kavya Reddy','Hyderabad','Laptop',1,78000,'Placed');

num_affected_rows,num_inserted_rows
2,2


In [0]:
insert into ecommerce_orders values
(112,'Santhosh','Hyderabad','Headphones',2,2500,'Shipped');

num_affected_rows,num_inserted_rows
1,1


In [0]:
insert into ecommerce_orders values
(113,'Suresh','Chennai','Headphones',5,2000,'Placed');

num_affected_rows,num_inserted_rows
1,1


In [0]:
insert into ecommerce_orders values
(114,'Jason','Hyderabad','Mobile',1,28000,'Placed'),
(115,'Yadav','Hyderabad','Tablet',2,25000,'Placed'),
(116,'Abdullah','Hyderabad','Laptop',1,78000,'Placed');

num_affected_rows,num_inserted_rows
3,3


**PART 2 — UPDATE EXERCISES**

In [0]:
update ecommerce_orders
set order_status = 'Shipped'
where order_id = 101;

num_affected_rows
1


In [0]:
update ecommerce_orders
set quantity = quantity + 1
where order_id = 102;

num_affected_rows
1


In [0]:
update ecommerce_orders
set price = 78000
where product = 'Laptop';

num_affected_rows
6


In [0]:
update ecommerce_orders
set city = 'Secunderabad'
where customer_name = 'Amit Sharma';

num_affected_rows
1


In [0]:
update ecommerce_orders
set order_status = 'Delivered'
where product = 'Mobile';

num_affected_rows
5


In [0]:
update ecommerce_orders
set price = 2500
where product = 'Headphones';

num_affected_rows
3


In [0]:
update ecommerce_orders
set price = price + 1000
where product = 'Tablet';

num_affected_rows
3


In [0]:
update ecommerce_orders
set order_status = 'Processing'
where city = 'Bangalore';

num_affected_rows
2


In [0]:
update ecommerce_orders
set quantity = 2
where quantity = 1;

num_affected_rows
12


In [0]:
update ecommerce_orders
set city = 'Surat'
where city = 'Ahmedabad';

num_affected_rows
1


**PART 3 — DELETE EXERCISES**

In [0]:
delete from ecommerce_orders
where order_status = 'Cancelled';

num_affected_rows
1


In [0]:
delete from ecommerce_orders
where quantity > 3;

num_affected_rows
1


In [0]:
delete from ecommerce_orders
where product = 'Camera';

num_affected_rows
1


In [0]:
delete from ecommerce_orders
where city = 'Kolkata';

num_affected_rows
1


In [0]:
delete from ecommerce_orders
where price < 5000;

num_affected_rows
2


In [0]:
delete from ecommerce_orders
where customer_name like 'A%';

num_affected_rows
4


In [0]:
delete from ecommerce_orders
where product = 'Tablet';

num_affected_rows
2


In [0]:
delete from ecommerce_orders
where city = 'Mumbai' and quantity = 1;

num_affected_rows
0


In [0]:
delete from ecommerce_orders
where price > 80000;

num_affected_rows
0


In [0]:
delete from ecommerce_orders
where order_id = (select max(order_id) from ecommerce_orders);

num_affected_rows
1


**PART 4 — MERGE / UPSERT EXERCISES**

**Creating incoming dataset**

In [0]:
create or replace temp view incoming_orders as
select * from values
(102,'Priya Reddy','Bangalore','Mobile',4,30000,'Delivered'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Shipped'),
(109,'Rahul Verma','Mumbai','Tablet',2,26000,'Placed'),
(110,'Divya Menon','Kochi','Mobile',1,27000,'Placed'),
(111,'Farhan Ali','Hyderabad','Laptop',1,79000,'Placed')
as incoming_orders(order_id,customer_name,city,product,quantity,price,order_status);

In [0]:
merge into ecommerce_orders as target
using incoming_orders as source
on target.order_id = source.order_id

when matched then
update set
  target.customer_name = source.customer_name,
  target.city = source.city,
  target.product = source.product,
  target.quantity = source.quantity,
  target.price = source.price,
  target.order_status = source.order_status

when not matched then
insert (
  order_id,
  customer_name,
  city,
  product,
  quantity,
  price,
  order_status
)
values (
  source.order_id,
  source.customer_name,
  source.city,
  source.product,
  source.quantity,
  source.price,
  source.order_status
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
6,4,0,2


In [0]:
select * from ecommerce_orders
order by order_id;

order_id,customer_name,city,product,quantity,price,order_status
102,Priya Reddy,Bangalore,Mobile,4,30000,Delivered
104,Sneha Iyer,Chennai,Laptop,1,72000,Shipped
108,Meera Nair,Bangalore,Laptop,2,78000,Processing
109,Rahul Verma,Mumbai,Tablet,2,26000,Placed
110,Divya Menon,Kochi,Mobile,1,27000,Placed
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed


In [0]:
select order_id,
case
when order_id in (102,104) then 'updated'
else 'inserted'
end as status
from ecommerce_orders;

order_id,status
108,inserted
102,updated
104,updated
109,inserted
110,inserted
111,inserted
111,inserted


In [0]:
merge into ecommerce_orders as target
using incoming_orders as source
on target.order_id = source.order_id
when matched then
update set target.order_status = source.order_status
when not matched then
insert *
;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
6,6,0,0


In [0]:
merge into ecommerce_orders as target
using incoming_orders as source
on target.order_id = source.order_id

when matched and source.order_status != 'Cancelled' then
update set target.order_status = source.order_status

when not matched and source.order_status != 'Cancelled' then
insert *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
6,6,0,0


In [0]:
merge into ecommerce_orders as target
using incoming_orders as source
on target.order_id = source.order_id

when not matched and source.product = 'Laptop' then
insert *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
merge into ecommerce_orders as target
using incoming_orders as source
on target.order_id = source.order_id

when matched and source.price > target.price then
update set target.price = source.price

when not matched then
insert *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


**PART 5 — REALISTIC UPSERT SCENARIO**

**Creating daily sales updates**

In [0]:
create or replace temp view daily_updates as
select * from values
(103,'Rohit Mehta','Mumbai','Headphones',5,2200,'Delivered'),
(106,'Ananya Das','Kolkata','Mobile',2,28000,'Shipped'),
(112,'Sanjay Gupta','Delhi','Tablet',1,24000,'Placed')
as daily_updates(order_id,customer_name,city,product,quantity,price,order_status);

In [0]:
merge into ecommerce_orders as target
using daily_updates as source
on target.order_id = source.order_id

when matched then
update set target.quantity = source.quantity

when not matched then
insert *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,3,0,0


In [0]:
merge into ecommerce_orders as target
using daily_updates as source
on target.order_id = source.order_id

when not matched then
insert *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,0,0,3


In [0]:
select count(*) from daily_updates
where order_id in (select order_id from ecommerce_orders);

count(*)
0


In [0]:
select count(*) from daily_updates
where order_id not in (select order_id from ecommerce_orders);

count(*)
3


In [0]:
select * from ecommerce_orders
order by price desc;

order_id,customer_name,city,product,quantity,price,order_status
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed
108,Meera Nair,Bangalore,Laptop,2,78000,Processing
104,Sneha Iyer,Chennai,Laptop,1,72000,Shipped
102,Priya Reddy,Bangalore,Mobile,4,30000,Delivered
110,Divya Menon,Kochi,Mobile,1,27000,Placed
109,Rahul Verma,Mumbai,Tablet,2,26000,Placed
